# From bigrams to GPT, step by step — problem sheet

This notebook accompanies the notes *From Bigrams to GPT* (Redes Neuronales 2026, FAMAF-UNC).
It implements the GPT of the notes **incrementally**: every section corresponds to one chapter, builds one model, and trains it with the same training loop, so that the validation loss can be compared from the simplest model to the full one.

| Section | Chapter of the notes | Model |
|---|---|---|
| 1 | Language modeling | uniform baseline |
| 2 | Embedding the context | (softmax, cross-entropy, token embeddings, training loop) |
| 3 | $n$-gram and bigram models | counting bigram, $n$-grams, neural bigram, bag of words |
| 4 | Self-attention | one attention head between embeddings and logits |
| 5 | The transformer block | one transformer block |
| 6 | Assembling GPT | positional embeddings, a stack of blocks, weight tying |
| 7 | Training and generation | full GPT, sampling with temperature and top-$k$ |

As in the notes, the architecture is built from primitives only: `nn.Embedding`, `nn.Linear`, `nn.LayerNorm`, matrix multiplication and softmax.
Gradients come from `loss.backward()` and parameter updates from `torch.optim.AdamW`, used as-is.

**How to work on this sheet.**

* Code exercises are marked `TODO (Exercise N)`. Replace `raise NotImplementedError(...)` by your implementation. Do not change function signatures or attribute names: later cells use them.
* Most exercises are followed by checks that print `PASS` or `FAIL`. Do not move on while a check fails: every section builds on the previous ones.
* Written exercises are followed by a cell *Answer*, where you write your answer (math in $\LaTeX$ is welcome).
* Exercises marked *(optional)* can be skipped without breaking the rest of the notebook.
* Code that is not marked `TODO` is given. Read it anyway: you will use it all the time.

**Running it.** On Google Colab, select a GPU runtime (*Runtime → Change runtime type*).
Locally, use the conda environment of `code/environment.yml` (`conda activate redes-2026`).
On a GPU the whole notebook takes a few minutes; on a CPU, Section 7 alone takes about 15–20 minutes.

## 0. Setup

In [ ]:
import math
import time
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.nn import functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", device)

# Validation loss of every model trained below, for the final comparison.
results = {}


def check(name, ok):
    # Prints PASS or FAIL for one test of an exercise.
    print(("PASS  " if bool(ok) else "FAIL  ") + name)

## 1. Language modeling (Chapter 1)

A language model assigns a probability to every sequence of tokens $x_1,\dots,x_T$.
By the chain rule, $p(x_1,\dots,x_T) = \prod_t p(x_t \mid x_{<t})$, so every model below is a way of parameterizing the conditional $p(x_t \mid x_{<t})$ of the next token given everything before it.

### 1.1 The corpus and the tokenizer

We use Tiny Shakespeare (about 1 MB of text), bundled in the repository as `code/data/tinyshakespeare.txt`.
If the file is not found (e.g. on Colab) it is downloaded.
Tokens are characters: the tokenizer is a bijection between the $V$ distinct characters of the corpus and the integers $0,\dots,V-1$ (`gpt_intro/data.py:CharTokenizer`).

In [ ]:
URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
candidates = [Path("../data/tinyshakespeare.txt"), Path("code/data/tinyshakespeare.txt"),
              Path("data/tinyshakespeare.txt"), Path("tinyshakespeare.txt")]
path = next((p for p in candidates if p.exists()), None)
if path is None:
    path = Path("tinyshakespeare.txt")
    urllib.request.urlretrieve(URL, path)
text = path.read_text(encoding="utf-8")
print(f"{path}: {len(text):,} characters")
print(text[:300])

In [ ]:
class CharTokenizer:
    # One token per character: x_t is the integer id of the t-th character.
    def __init__(self, text):
        chars = sorted(set(text))
        self.itos = {i: ch for i, ch in enumerate(chars)}
        self.stoi = {ch: i for i, ch in enumerate(chars)}
        self.vocab_size = len(chars)

    def encode(self, s):
        return [self.stoi[c] for c in s]

    def decode(self, ids):
        return "".join(self.itos[i] for i in ids)


tok = CharTokenizer(text)
V = tok.vocab_size
print("V =", V)
print(repr("".join(tok.itos[i] for i in range(V))))
print(tok.encode("To be, or not to be"))
print(tok.decode(tok.encode("To be, or not to be")))

### 1.2 Training and validation data

The corpus becomes one long tensor of token ids.
The first 90% is used for training and the last 10% for validation.
A training example is a chunk of $T$ consecutive tokens together with the same chunk shifted by one (Section *Teacher forcing and batching* of Chapter 7): the target at position $t$ is the token at position $t+1$.
A batch stacks $B$ such chunks, taken at random offsets, into tensors of shape $(B, T)$.

In [ ]:
ids = torch.tensor(tok.encode(text), dtype=torch.long)
n = int(0.9 * len(ids))
train_ids, val_ids = ids[:n], ids[n:]
print(f"train: {len(train_ids):,} tokens | val: {len(val_ids):,} tokens")


def get_batch(split, block_size, batch_size):
    # x = (x_i, ..., x_{i+T-1}),  y = (x_{i+1}, ..., x_{i+T})
    data = train_ids if split == "train" else val_ids
    i = torch.randint(0, len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[j : j + block_size] for j in i])
    y = torch.stack([data[j + 1 : j + block_size + 1] for j in i])
    return x.to(device), y.to(device)


x, y = get_batch("train", block_size=8, batch_size=2)
print(x)
print(y)

**Exercise 1.** A chunk of $T$ tokens `x` comes with its targets `y`.
How many next-token predictions, i.e. terms $\ell_t$ of the loss, does one chunk contribute?
What context does the prediction at position `t` of the chunk see?
Verify in code that `y[:, :-1]` equals `x[:, 1:]` for the batch printed above.

In [ ]:
# TODO (Exercise 1): check that y is x shifted by one position.

> **Answer:** *write your answer here.*

### 1.3 The loss, and a baseline that knows nothing

The loss is the average negative log-likelihood per token,
$$\mathcal L = -\frac1T\sum_{t=1}^T \log p(x_t \mid x_{<t}) = \frac1T\sum_{t=1}^T \ell_t,
\qquad \ell_t := -\log p(x_t\mid x_{<t}),$$
measured in nats per token. Its exponential is the perplexity, $\mathrm{PPL} = e^{\mathcal L}$.

A model that assigns the uniform distribution $p(x_t \mid x_{<t}) = 1/V$ to every token has $\mathcal L = \log V$ and $\mathrm{PPL} = V$.
This is the loss an untrained model should start from, and the number every model below must beat.

In [ ]:
def uniform_model_loss(V):
    # TODO (Exercise 2): average negative log-likelihood per token, in nats,
    # of the model that assigns probability 1/V to every token.
    raise NotImplementedError("Exercise 2")


uniform_loss = uniform_model_loss(V)
print(f"uniform model: loss = {uniform_loss:.4f} nats, perplexity = {math.exp(uniform_loss):.1f}")
check("uniform loss = log 65", abs(uniform_loss - math.log(65)) < 1e-9)
results["uniform"] = uniform_loss

**Exercise 2 (written part).** Show that the loss of the uniform model is $\log V$ for *any* sequence, and that its perplexity is $V$.
Why is $\log V$ also, approximately, the loss of an untrained model whose logits are initialized close to zero?

> **Answer:** *write your answer here.*

## 2. Embedding the context (Chapter 2)

Every model in the notes has the same form,
$$x_1,\dots,x_{t-1} \;\longmapsto\; e_{x_1},\dots,e_{x_{t-1}} \;\longmapsto\; c_{t-1}\in\mathbb R^d \;\longmapsto\; z_t = W c_{t-1}\in\mathbb R^V,$$
and the models differ only in the middle arrow, i.e. in how the context vector $c_{t-1}$ is computed.
Representations are indexed by the last token they have read, and logits by the token they predict.

**In code**, a model takes a batch of token ids `idx` of shape $(B, T)$ and returns `logits` of shape $(B, T, V)$.
Position `t` of the chunk has read `idx[:, :t+1]`, so `logits[:, t]` scores the candidates for the next token, which is `targets[:, t]`.

This section checks the formulas of Chapter 2 numerically, and defines the training loop used by every later model.

### 2.1 Softmax
$$\sigma_b(z) = \frac{e^{z_b}}{\sum_{b'} e^{z_{b'}}}$$
is non-negative, sums to one, and is shift-invariant: $\sigma(z + \kappa\mathbf 1) = \sigma(z)$.

In [ ]:
def softmax(z):
    # TODO (Exercise 3): softmax along the last dimension of z.
    # It must be numerically stable: exp(1000.) overflows in float32.
    # Hint: use shift invariance. Do not call F.softmax or torch.softmax.
    raise NotImplementedError("Exercise 3")


z = torch.randn(V)
p = softmax(z)
check("sums to one", torch.allclose(p.sum(), torch.tensor(1.0)))
check("matches F.softmax", torch.allclose(p, F.softmax(z, dim=-1)))
check("shift invariant", torch.allclose(p, softmax(z + 123.4)))
check("no overflow for large logits", torch.isfinite(softmax(torch.tensor([1000.0, 0.0]))).all())
check("works on batches", torch.allclose(softmax(torch.ones(3, 4)), torch.full((3, 4), 0.25)))

### 2.2 Cross-entropy and its gradient

With $p_\theta(x_t = b \mid x_{<t}) = \sigma_b(z_t)$, the per-position loss is the cross-entropy against the observed token,
$$\ell_t = -\log\sigma_{x_t}(z_t) = -(z_t)_{x_t} + \log\sum_b e^{(z_t)_b},$$
and its gradient with respect to the logits is
$$\frac{\partial \ell_t}{\partial (z_t)_b} = \sigma_b(z_t) - \delta_{b,x_t}.$$
`F.cross_entropy` takes raw logits and integer targets, and averages $\ell_t$ over all positions.

**Exercise 4.**
(a) Derive the gradient formula above from the expression of $\ell_t$ in terms of the logits.
(b) Implement $\ell_t$ by hand and the gradient formula, and compare them with `F.cross_entropy` and with autograd.

> **Answer:** *write your answer here.*

In [ ]:
def per_position_loss(z, targets):
    # TODO (Exercise 4b): z has shape (N, V), targets shape (N,).
    # Return the N losses l_t = -(z_t)_{x_t} + log sum_b exp((z_t)_b).
    # Hint: torch.logsumexp. Do not call F.cross_entropy.
    raise NotImplementedError("Exercise 4b")


def mean_loss_gradient(z, targets):
    # TODO (Exercise 4b): gradient of the *mean* of the N losses with
    # respect to z, shape (N, V), computed from the formula, not with autograd.
    raise NotImplementedError("Exercise 4b")


z = torch.randn(5, V, requires_grad=True)       # logits at 5 positions
targets = torch.randint(0, V, (5,))             # observed tokens

loss = per_position_loss(z, targets).mean()
check("by hand == F.cross_entropy", torch.allclose(loss, F.cross_entropy(z, targets)))
loss.backward()
check("formula == autograd gradient",
      torch.allclose(z.grad, mean_loss_gradient(z.detach(), targets), atol=1e-6))

### 2.3 Token embeddings

An embedding table $E\in\mathbb R^{V\times d}$ stores one learnable vector $e_a$ (row $a$) per token.
A lookup is indexing, but it equals $E^\top u_a$, with $u_a$ the one-hot vector of token $a$: an embedding is a linear layer applied to one-hot inputs, implemented efficiently.

**Exercise 5.** Compute the same rows as a product with one-hot vectors (`F.one_hot`), without indexing `emb.weight`.

In [ ]:
def embed_via_one_hot(a, E):
    # TODO (Exercise 5): a is a tensor of token ids, E the (V, d) embedding table.
    # Return the rows e_a computed as one-hot vectors times E.
    raise NotImplementedError("Exercise 5")


d = 16
emb = nn.Embedding(V, d)                         # E, initialized at random
a = torch.tensor([3, 17, 42])
lookup = emb(a)                                  # rows e_a
check("one-hot product == lookup", torch.allclose(lookup, embed_via_one_hot(a, emb.weight)))

### 2.4 The training loop and sampling

The same three functions are used for every model below (Chapter 7):

* `estimate_loss` averages the loss over a few random batches of each split, without gradients;
* `train` repeats: sample a batch, compute the loss, `loss.backward()`, `optimizer.step()` with AdamW;
* `generate` samples autoregressively: predict the next token, append it, and feed the extended sequence back in, cropped to the last `block_size` tokens. `temperature` and `top_k` are explained in Section 7.

These functions are given. Read them carefully: every model below is trained and sampled with them.

In [ ]:
def lm_loss(logits, targets):
    B, T, V_ = logits.shape
    return F.cross_entropy(logits.view(B * T, V_), targets.view(B * T))


@torch.no_grad()
def estimate_loss(model, block_size, batch_size=64, eval_iters=50):
    model.eval()
    out = {}
    for split in ("train", "val"):
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            x, y = get_batch(split, block_size, batch_size)
            losses[k] = lm_loss(model(x), y).item()
        out[split] = losses.mean().item()
    model.train()
    return out


def train(model, *, block_size, max_iters, lr, batch_size=64, eval_interval=None, seed=1337):
    torch.manual_seed(seed)
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    eval_interval = eval_interval or max(1, max_iters // 5)
    history = []
    t0 = time.time()
    for it in range(1, max_iters + 1):
        x, y = get_batch("train", block_size, batch_size)
        loss = lm_loss(model(x), y)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        if it % eval_interval == 0 or it == max_iters:
            l = estimate_loss(model, block_size)
            history.append((it, l["train"], l["val"]))
            print(f"step {it:5d} | train loss {l['train']:.4f} | val loss {l['val']:.4f} | {time.time() - t0:5.1f}s")
    return history


@torch.no_grad()
def generate(model, max_new_tokens, block_size, prompt="\n", temperature=1.0, top_k=None, seed=0):
    torch.manual_seed(seed)
    model.eval()
    idx = torch.tensor([tok.encode(prompt)], device=device)
    for _ in range(max_new_tokens):
        logits = model(idx[:, -block_size:])[:, -1, :] / temperature   # z_{t+1}
        if top_k is not None:
            kth = torch.topk(logits, top_k).values[:, [-1]]
            logits = logits.masked_fill(logits < kth, float("-inf"))
        next_id = torch.multinomial(F.softmax(logits, dim=-1), num_samples=1)
        idx = torch.cat([idx, next_id], dim=1)
    model.train()
    return tok.decode(idx[0].tolist())


def num_params(model):
    return sum(p.numel() for p in model.parameters())

**Exercise 6.**
(a) In `train`, what would go wrong if the line `optimizer.zero_grad(set_to_none=True)` were removed?
(b) In `generate`, why is the input cropped to `idx[:, -block_size:]`, and why is only the last position of the logits, `[:, -1, :]`, used?

> **Answer:** *write your answer here.*

## 3. $n$-gram and bigram models (Chapter 3)

The $n$-gram approximation keeps only the last $n-1$ tokens of the context, $p(x_t\mid x_{<t}) \approx p(x_t \mid x_{t-n+1},\dots,x_{t-1})$.
For a fixed $n$ the contexts can be enumerated, and the model is a table of conditional probabilities, fitted exactly by counting.
The bigram, $n = 2$, truncates the context to the last token, $p(x_t\mid x_{<t}) \approx p(x_t \mid x_{t-1})$.
The model is then a $V\times V$ table $P_{ab} = p(x_t = b \mid x_{t-1} = a)$.

### 3.1 Fitting the bigram table by counting

The maximum-likelihood estimate is $\hat P_{ab} = N_{ab}/\sum_{b'}N_{ab'}$, where $N_{ab}$ counts how often token $b$ follows token $a$ in the corpus (derived in Section *The case $n = 2$: bigrams* of the notes, from the decomposition of the cross-entropy into entropy plus KL divergence).
With additive (Laplace) smoothing, a constant $\nu > 0$ is added to every count, so that pairs never seen in training do not get probability zero.
Normalizing each row, the smoothed estimate is $(N_{ab}+\nu)/(\sum_{b'}N_{ab'} + V\nu)$.

**Exercise 7.** Implement the counts, the smoothed table and its loss.

In [ ]:
def bigram_counts(data):
    # TODO (Exercise 7a): float tensor of shape (V, V) with N[a, b] = number of
    # times token b follows token a in data (a 1-D tensor of token ids).
    # Hint: a Python loop over 10^6 pairs is slow; torch.bincount of a * V + b is fast.
    raise NotImplementedError("Exercise 7a")


def smoothed_probs(N, nu):
    # TODO (Exercise 7b): Laplace-smoothed estimate (N_ab + nu) / (sum_b' N_ab' + V nu).
    raise NotImplementedError("Exercise 7b")


def counting_nll(P, data):
    # TODO (Exercise 7c): average negative log-likelihood of data under the
    # bigram table P, over the pairs (x_{t-1}, x_t), t = 2, ..., T.
    raise NotImplementedError("Exercise 7c")


N = bigram_counts(train_ids)
nu = 1.0
P_hat = smoothed_probs(N, nu)
train_nll, val_nll = counting_nll(P_hat, train_ids), counting_nll(P_hat, val_ids)
print(f"counting bigram | train loss {train_nll:.4f} | val loss {val_nll:.4f}")
check("counts every consecutive pair once", N.sum().item() == len(train_ids) - 1)
check("rows of P_hat sum to one", torch.allclose(P_hat.sum(dim=1), torch.ones(V)))
check("train loss close to 2.45 nats (notes)", abs(train_nll - 2.45) < 0.01)
check("val loss close to 2.48 nats (notes)", abs(val_nll - 2.48) < 0.01)
results["counting bigram"] = val_nll

# Most likely successors of a few characters
for ch in ["q", "T", "\n"]:
    top = torch.topk(P_hat[tok.stoi[ch]], 3)
    print(repr(ch), "->", [(tok.itos[i], round(p, 3)) for p, i in zip(top.values.tolist(), top.indices.tolist())])

In [ ]:
# Sampling from the counting bigram: x_t ~ P_hat[x_{t-1}]
torch.manual_seed(0)
idx = [tok.stoi["\n"]]
for _ in range(300):
    idx.append(torch.multinomial(P_hat[idx[-1]], 1).item())
print(tok.decode(idx))

**Exercise 8.** Look at the sample above. Which features of English does it reproduce, and which does it miss?
Explain both in terms of the bigram approximation.

> **Answer:** *write your answer here.*

### 3.2 Longer contexts: $n$-grams (optional)

**Exercise 9 (optional).** Generalize the counting model to an arbitrary $n$, as suggested in the notes.
The full table has $V^{n-1}$ rows ($65^5 \approx 10^9$ for $n = 6$), so store only the observed counts, in a dictionary keyed by tuples of token ids (`collections.Counter`).
As in the notes, leave out the first $n-1$ positions, which have no full context, and evaluate the loss on positions $t = n, \dots, T$.
Return `float("inf")` when some evaluated token gets probability zero.

For reference, the values in the notes are (losses in nats per token):

| $n$ | train, $\nu=0$ | train, $\nu=1$ | val, $\nu=1$ |
|---|---|---|---|
| 1 | 3.31 | 3.31 | 3.35 |
| 2 | 2.45 | 2.45 | 2.48 |
| 3 | 1.90 | 1.95 | 2.07 |
| 4 | 1.49 | 1.74 | 1.95 |
| 5 | 1.22 | 1.88 | 2.17 |
| 6 | 0.98 | 2.20 | 2.59 |

Then answer:
(a) Which $n$ gives the best validation loss? Why does the validation loss grow for larger $n$, while the unsmoothed training loss keeps falling?
(b) What is the unsmoothed ($\nu = 0$) validation loss, and why?
(c) Why does smoothing with $\nu = 1$ make the *training* loss grow for large $n$?

In [ ]:
from collections import Counter


def ngram_losses(n, nu, train, val):
    # TODO (Exercise 9, optional): fit an n-gram model by counting on the list
    # of token ids `train`, with Laplace smoothing nu, and return the pair
    # (train_loss, val_loss), each averaged over positions t = n, ..., T.
    raise NotImplementedError("Exercise 9")


train_list, val_list = train_ids.tolist(), val_ids.tolist()
for n in range(1, 7):
    tr0, va0 = ngram_losses(n, 0.0, train_list, val_list)
    tr1, va1 = ngram_losses(n, 1.0, train_list, val_list)
    print(f"n = {n} | train (nu=0) {tr0:.2f} | train (nu=1) {tr1:.2f} | val (nu=1) {va1:.2f} | val (nu=0) {va0:.2f}")

> **Answer:** *write your answer here.*

### 3.3 The same model, trained by gradient descent

In the language of Chapter 2, the bigram model is the case in which the context vector is the token embedding of the last token, $c_{t-1} = e_{x_{t-1}}$, with $d = V$ and $W = I$, so that the token embedding is used directly as logits.
Calling the embedding table $Z\in\mathbb R^{V\times V}$, the logits for $x_t$ are row $x_{t-1}$ of $Z$,
$$z_t = Z_{x_{t-1}}, \qquad p_Z(x_t = b\mid x_{t-1} = a) = \sigma_b(Z_a).$$
Training $Z$ with cross-entropy should reach the same loss as counting, since both maximize the same likelihood (`gpt_intro/bigram.py:NeuralBigramLM`).

**Exercise 10.** Implement the forward pass of the neural bigram.

In [ ]:
class NeuralBigram(nn.Module):
    def __init__(self, V):
        super().__init__()
        self.Z = nn.Embedding(V, V)     # row a = logits for the token after a

    def forward(self, idx):             # idx: (B, T)
        # TODO (Exercise 10): return the logits, shape (B, T, V), with
        # logits[:, t] = Z[x_t], the scores of the candidates for x_{t+1}.
        raise NotImplementedError("Exercise 10")


bigram = NeuralBigram(V)
print("parameters:", num_params(bigram))
# The loss is convex in Z, so a large learning rate is fine here.
train(bigram, block_size=64, max_iters=3000, lr=3e-2)
results["neural bigram"] = estimate_loss(bigram, block_size=64)["val"]
check("reaches the loss of the counting bigram",
      abs(results["neural bigram"] - results["counting bigram"]) < 0.05)
print(generate(bigram, 300, block_size=1))

In [ ]:
# The learned table, turned into probabilities, approaches the counting estimate.
P_neural = F.softmax(bigram.Z.weight.detach().cpu(), dim=-1)
print(f"neural table evaluated as a counting table: val loss {counting_nll(P_neural, val_ids):.4f}")

fig, axs = plt.subplots(1, 2, figsize=(10, 4.5))
for ax, M, title in zip(axs, [P_hat, P_neural], [r"counting $\hat P$", r"gradient descent $\sigma(Z_a)$"]):
    ax.imshow(M.log(), cmap="viridis")
    ax.set_title(title + " (log scale)")
    ax.set_xlabel("next token $b$")
    ax.set_ylabel("previous token $a$")
plt.tight_layout()
plt.show()

**Exercise 11.**
(a) Why can the neural bigram not reach a lower *training* loss than the unsmoothed counting bigram, however long it is trained?
(b) Compare the two panels. Where do they differ most, and why? (Hint: consider the pairs $(a, b)$ that never occur in the training split. How does each method treat them: smoothing for counting, and the gradient $\sigma_b(z) - \delta_{b,x_t}$ for gradient descent?)

> **Answer:** *write your answer here.*

### 3.4 Using the whole prefix: bag of words

A bigram model only ever looks at $x_{t-1}$.
The crudest way to use the whole prefix is to average the embeddings of all the tokens read so far, $c_t = \frac1t\sum_{s\le t} e_{x_s}$, and map the average to logits with $z_{t+1} = W c_t$, $W\in\mathbb R^{V\times d}$.

For all positions at once, the averages are a matrix product: stacking the embeddings as the rows of $H\in\mathbb R^{T\times d}$, the context vectors are the rows of $AH$, where $A$ is lower-triangular with $A_{ts} = 1/t$ for $s \le t$.
Note that $A$ is also what a row-wise softmax gives when applied to a matrix of zeros with the causal mask ($-\infty$ above the diagonal): attention, in the next section, replaces those zeros by learned, content-dependent scores.
(In code, $t$ counts positions inside the chunk, so the context is limited to the last `block_size` tokens.)

**Exercise 12.** Build the averaging matrix $A$ and use it in the bag-of-words model.

In [ ]:
def averaging_matrix(T):
    # TODO (Exercise 12a): (T, T) lower-triangular matrix whose row t averages
    # positions 0..t uniformly (0-based indices). Hint: torch.tril.
    raise NotImplementedError("Exercise 12a")


T = 5
A = averaging_matrix(T)
print(A)
mask = torch.tril(torch.ones(T, T))
scores = torch.zeros(T, T).masked_fill(mask == 0, float("-inf"))
check("rows sum to one", torch.allclose(A.sum(dim=1), torch.ones(T)))
check("lower-triangular", torch.equal(A, torch.tril(A)))
check("A == softmax(masked zeros)", torch.allclose(A, F.softmax(scores, dim=-1)))

In [ ]:
class BagOfWords(nn.Module):
    def __init__(self, V, d, block_size):
        super().__init__()
        self.E = nn.Embedding(V, d)
        self.W = nn.Linear(d, V, bias=False)
        self.register_buffer("A", averaging_matrix(block_size))   # not a parameter

    def forward(self, idx):
        # TODO (Exercise 12b): embed the tokens, average them with A (cropped to
        # the length T of the input), and map the averages to logits with W.
        # Shapes: idx (B, T) -> H (B, T, d) -> C (B, T, d) -> logits (B, T, V).
        raise NotImplementedError("Exercise 12b")


bow = BagOfWords(V, d=64, block_size=64)
print("parameters:", num_params(bow))
train(bow, block_size=64, max_iters=3000, lr=3e-3)
results["bag of words"] = estimate_loss(bow, block_size=64)["val"]

**Exercise 13.** Compare the validation loss of the bag-of-words model with that of the bigram.
The bag of words uses the *whole* prefix and the bigram only the last token: explain the result.
What would the averaging weights need in order to do better?

> **Answer:** *write your answer here.*

## 4. Self-attention (Chapter 4)

### 4.1 Queries, keys, values, and the attention weights

Each position $s$ produces a query, a key and a value from its embedding,
$$q_s = W_Q^\top e_{x_s},\quad k_s = W_K^\top e_{x_s},\quad v_s = W_V^\top e_{x_s}\in\mathbb R^{d_k},$$
and the context vector at position $t$ is the weighted average
$$c_t = \sum_{s\le t}\alpha_{ts}\,v_s, \qquad \alpha_{ts} = \sigma_{s\le t}\!\left(\frac{q_t\cdot k_s}{\sqrt{d_k}}\right).$$
In matrix form, with $Q = HW_Q$, $K = HW_K$, $V = HW_V$,
$$C = \mathrm{Attention}(Q,K,V) = \sigma\!\left(\frac{QK^\top}{\sqrt{d_k}} + M\right)V,$$
where $M_{ts} = 0$ for $s\le t$ and $-\infty$ for $s>t$ is the causal mask, and row $t$ of $C$ is $c_t^\top$.
We first check that the two forms agree.

**Exercise 14.** Implement the matrix form. The scalar form, one position at a time, is given and used as a check.

In [ ]:
def attention(Q, K, V, causal=True):
    # TODO (Exercise 14): Q, K, V have shape (..., T, d_k), with any number of
    # leading batch dimensions. Return C = softmax(Q K^T / sqrt(d_k) + M) V,
    # shape (..., T, d_k), with the causal mask M applied only if causal=True.
    # Hint: torch.tril and masked_fill with float("-inf").
    raise NotImplementedError("Exercise 14")


torch.manual_seed(0)
T, d, d_k = 6, 16, 8
H = torch.randn(T, d)
W_Q, W_K, W_V = torch.randn(d, d_k), torch.randn(d, d_k), torch.randn(d, d_k)
Q, K, Vm = H @ W_Q, H @ W_K, H @ W_V          # rows q_s^T, k_s^T, v_s^T

# Scalar form: one position at a time
c = torch.zeros(T, d_k)
for t in range(T):
    s_le_t = torch.arange(t + 1)
    alpha = softmax(torch.stack([Q[t] @ K[s] / math.sqrt(d_k) for s in s_le_t]))
    c[t] = sum(alpha[s] * Vm[s] for s in s_le_t)

check("scalar form == matrix form", torch.allclose(c, attention(Q, K, Vm), atol=1e-5))
check("works with batch dimensions",
      torch.allclose(attention(Q[None, None], K[None, None], Vm[None, None])[0, 0], c, atol=1e-5))
check("causal: position 0 only sees itself", torch.allclose(attention(Q, K, Vm)[0], Vm[0], atol=1e-6))

### 4.2 Why the $\sqrt{d_k}$

If the components of $q_t$ and $k_s$ are independent with mean 0 and variance 1, the dot product $q_t\cdot k_s$ has variance $d_k$.
Without the scaling, softmax saturates for large $d_k$: one weight takes almost all the mass.

**Exercise 15.** Prove the claim: if the $2 d_k$ components of $q_t$ and $k_s$ are independent, with mean $0$ and variance $1$, then $q_t\cdot k_s$ has mean $0$ and variance $d_k$.
Then check it against the numbers printed below.

> **Answer:** *write your answer here.*

In [ ]:
torch.manual_seed(0)
for d_k in [4, 64, 1024]:
    q, k = torch.randn(d_k), torch.randn(1000, d_k)
    raw = k @ q
    a_raw, a_scaled = F.softmax(raw[:10], -1), F.softmax(raw[:10] / math.sqrt(d_k), -1)
    print(f"d_k={d_k:5d} | var(q.k) = {raw.var():8.1f} | max weight, unscaled {a_raw.max():.3f} | scaled {a_scaled.max():.3f}")

### 4.3 An attention language model

A single attention head between the embeddings and the logits is already a complete language model (Eq. *attention-lm* of the notes):
$$x_{\le t}\;\to\; e_{x_s}\;\to\; q_s,k_s,v_s\;\to\; c_t\;\to\; z_{t+1} = W c_t,\qquad W\in\mathbb R^{V\times d_k}.$$
The head is `gpt_intro/attention.py:CausalSelfAttentionHead`: three bias-free linear layers for $W_Q, W_K, W_V$ and the masked softmax.
There are no positional embeddings yet; they come in Section 6.

**Exercise 16.** Implement the forward passes of the head and of the language model.

In [ ]:
class Head(nn.Module):
    # One causal self-attention head: H (B, T, d) -> C (B, T, d_k)
    def __init__(self, d, d_k):
        super().__init__()
        self.W_Q = nn.Linear(d, d_k, bias=False)
        self.W_K = nn.Linear(d, d_k, bias=False)
        self.W_V = nn.Linear(d, d_k, bias=False)

    def forward(self, H):
        # TODO (Exercise 16a): queries, keys and values, then attention.
        raise NotImplementedError("Exercise 16a")


class AttentionLM(nn.Module):
    def __init__(self, V, d, d_k):
        super().__init__()
        self.E = nn.Embedding(V, d)
        self.head = Head(d, d_k)
        self.W = nn.Linear(d_k, V, bias=False)

    def forward(self, idx):
        # TODO (Exercise 16b): embeddings -> head -> logits.
        raise NotImplementedError("Exercise 16b")


attn_lm = AttentionLM(V, d=64, d_k=64)
print("parameters:", num_params(attn_lm))
train(attn_lm, block_size=64, max_iters=3000, lr=3e-3)
results["one attention head"] = estimate_loss(attn_lm, block_size=64)["val"]
print(generate(attn_lm, 300, block_size=64))

Let us look at the attention weights $\alpha_{ts}$ of the trained head on a short piece of text.
Row $t$ shows how the context vector $c_t$ distributes its weight over the positions $s \le t$; everything above the diagonal is masked.
Without positional information, the head can only select tokens by their content.

In [ ]:
@torch.no_grad()
def attention_weights(head, H):
    # With V = identity, attention returns softmax(Q K^T / sqrt(d_k) + M) @ I,
    # i.e. the matrix of attention weights alpha itself.
    Q, K = head.W_Q(H), head.W_K(H)
    T = Q.shape[-2]
    return attention(Q, K, torch.eye(T, device=H.device))


snippet = "First Citizen:\nBefore we"
idx = torch.tensor([tok.encode(snippet)], device=device)
alpha = attention_weights(attn_lm.head, attn_lm.E(idx))[0].cpu()
labels = [repr(ch)[1:-1] for ch in snippet]
plt.figure(figsize=(6, 6))
plt.imshow(alpha, cmap="Blues")
plt.xticks(range(len(snippet)), labels)
plt.yticks(range(len(snippet)), labels)
plt.xlabel("attended position $s$")
plt.ylabel("query position $t$")
plt.title(r"attention weights $\alpha_{ts}$")
plt.show()

**Exercise 17.** Describe the pattern of the attention weights in the plot.
Can this head single out "the previous token" or "the token three positions back"? Why not?
What does this imply for its loss, compared with the bigram's?

> **Answer:** *write your answer here.*

## 5. The transformer block (Chapter 5)

From now on, the vector at position $t$ is a hidden representation $h_t\in\mathbb R^d$ that is updated in place as it flows through the network, starting from $h_t = e_{x_t}$.

### 5.1 Multi-head attention, naive and fast

Multi-head attention runs $n_h$ heads of size $d_k = d/n_h$ in parallel, concatenates their outputs and mixes them with $W_O\in\mathbb R^{d\times d}$:
$$\mathrm{MHA}(H) = \big[\mathrm{Attention}_1(H)\,\Vert\,\cdots\,\Vert\,\mathrm{Attention}_{n_h}(H)\big]\,W_O.$$
`MultiHeadAttentionNaive` follows this definition with a Python loop over heads.
`MultiHeadAttentionFast` computes $Q,K,V$ for all heads with a single linear layer and treats the head index as a batch dimension, as nanoGPT and GPT-2 do (`gpt_intro/attention.py`).
With the same weights, both compute the same function.

**Exercise 18.** Implement both forward passes. The check copies the weights of the naive version into the fast one, so both must compute the same function.

In [ ]:
class MultiHeadAttentionNaive(nn.Module):
    def __init__(self, d, n_h, dropout=0.0):
        super().__init__()
        assert d % n_h == 0
        self.heads = nn.ModuleList([Head(d, d // n_h) for _ in range(n_h)])
        self.W_O = nn.Linear(d, d)
        self.dropout = nn.Dropout(dropout)

    def forward(self, H):
        # TODO (Exercise 18a): run every head, concatenate the outputs along the
        # feature dimension, apply W_O and then dropout.
        raise NotImplementedError("Exercise 18a")


class MultiHeadAttentionFast(nn.Module):
    def __init__(self, d, n_h, dropout=0.0):
        super().__init__()
        assert d % n_h == 0
        self.n_h, self.d_k = n_h, d // n_h
        self.W_QKV = nn.Linear(d, 3 * d, bias=False)   # W_Q, W_K, W_V of all heads
        self.W_O = nn.Linear(d, d)
        self.dropout = nn.Dropout(dropout)

    def forward(self, H):
        # TODO (Exercise 18b): with one call to W_QKV, compute Q, K, V of all heads,
        # each of shape (B, T, d). Reshape each to (B, n_h, T, d_k), so that the head
        # index acts as a batch dimension, call `attention` once, put the heads back
        # side by side, shape (B, T, d), and apply W_O and dropout.
        # Hint: .split, .view, .transpose, .reshape.
        raise NotImplementedError("Exercise 18b")


# Copy the weights of the naive version into the fast one and compare.
torch.manual_seed(0)
d, n_h = 32, 4
naive, fast = MultiHeadAttentionNaive(d, n_h), MultiHeadAttentionFast(d, n_h)
with torch.no_grad():
    for i, head in enumerate(naive.heads):
        rows = slice(i * fast.d_k, (i + 1) * fast.d_k)
        for j, W in enumerate([head.W_Q, head.W_K, head.W_V]):    # q, k, v blocks of W_QKV
            fast.W_QKV.weight[j * d : (j + 1) * d][rows] = W.weight
    fast.W_O.load_state_dict(naive.W_O.state_dict())
H = torch.randn(2, 10, d)
check("naive == fast", torch.allclose(naive(H), fast(H), atol=1e-6))

### 5.2 LayerNorm

$$\mathrm{LayerNorm}(h) = \gamma\odot\frac{h-\mu(h)}{\sqrt{\mathrm{Var}(h)+\epsilon}} + \beta,$$
with the mean $\mu(h)$ and the (biased) variance $\mathrm{Var}(h)$ taken over the $d$ components of each $h$ separately, and $\gamma,\beta\in\mathbb R^d$ learned.
Note that $\mu(h)$ and $\mathrm{Var}(h)$ are *scalars*, the same for every component of $h$, while $\gamma_i$ and $\beta_i$ depend on the component $i$.

**Exercise 19.** Implement LayerNorm by hand and compare it with `nn.LayerNorm`.

In [ ]:
def layernorm_by_hand(H, gamma, beta, eps):
    # TODO (Exercise 19): normalize every vector h along the last dimension of H.
    # Careful: nn.LayerNorm uses the biased variance (divide by d, not d - 1).
    raise NotImplementedError("Exercise 19")


ln = nn.LayerNorm(d)
with torch.no_grad():
    ln.weight.normal_(), ln.bias.normal_()      # non-trivial gamma, beta for the check
by_hand = layernorm_by_hand(H, ln.weight, ln.bias, ln.eps)
check("by hand == nn.LayerNorm", torch.allclose(by_hand, ln(H), atol=1e-5))

### 5.3 The feed-forward network

$$\mathrm{FFN}(h) = W_2\,\phi(W_1 h + b_1) + b_2,\qquad W_1\in\mathbb R^{4d\times d},\ W_2\in\mathbb R^{d\times 4d},$$
applied independently at every position, with $\phi = \mathrm{GELU}$, $\mathrm{GELU}(x) = x\,\Phi(x)$ and $\Phi$ the standard normal CDF.

**Exercise 20.** Implement GELU from its definition, and the feed-forward network.

In [ ]:
def gelu(x):
    # TODO (Exercise 20a): x * Phi(x), with Phi the standard normal CDF.
    # Hint: Phi(x) = (1 + erf(x / sqrt(2))) / 2, see torch.erf.
    raise NotImplementedError("Exercise 20a")


xs = torch.linspace(-4, 4, 201)
check("x Phi(x) == nn.GELU", torch.allclose(gelu(xs), nn.GELU()(xs), atol=1e-6))
plt.figure(figsize=(5, 3))
plt.plot(xs, F.relu(xs), label="ReLU")
plt.plot(xs, gelu(xs), label="GELU")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


class FFN(nn.Module):
    def __init__(self, d, dropout=0.0):
        super().__init__()
        self.W_1 = nn.Linear(d, 4 * d)
        self.W_2 = nn.Linear(4 * d, d)
        self.dropout = nn.Dropout(dropout)

    def forward(self, h):
        # TODO (Exercise 20b): W_1, GELU, W_2, then dropout.
        raise NotImplementedError("Exercise 20b")


ffn = FFN(8)
h = torch.randn(2, 5, 8)
check("FFN keeps the shape", ffn(h).shape == h.shape)
check("FFN acts on each position separately", torch.allclose(ffn(h)[:, 3], ffn(h[:, 3:4])[:, 0], atol=1e-6))

### 5.4 Residual connections and the full block

Each sub-layer reads a LayerNorm'd copy of the running representation and *adds* its output to it (pre-norm, as in GPT-2):
$$h \leftarrow h + \mathrm{MHA}(\mathrm{LayerNorm}(h)),\qquad h \leftarrow h + \mathrm{FFN}(\mathrm{LayerNorm}(h)).$$
The attention output is now a correction to $h_t$, and the map to logits moves to the end: $z_{t+1} = W h_t$.
We first train a model with a single block and still no positional embeddings.

**Exercise 21.** Implement the forward pass of the block.

In [ ]:
class Block(nn.Module):
    def __init__(self, d, n_h, dropout=0.0):
        super().__init__()
        self.ln_1 = nn.LayerNorm(d)
        self.mha = MultiHeadAttentionFast(d, n_h, dropout)
        self.ln_2 = nn.LayerNorm(d)
        self.ffn = FFN(d, dropout)

    def forward(self, h):
        # TODO (Exercise 21): the two pre-norm residual updates.
        raise NotImplementedError("Exercise 21")


class BlockLM(nn.Module):
    def __init__(self, V, d, n_h):
        super().__init__()
        self.E = nn.Embedding(V, d)
        self.block = Block(d, n_h)
        self.W = nn.Linear(d, V, bias=False)

    def forward(self, idx):
        h = self.E(idx)            # h_t = e_{x_t}
        h = self.block(h)          # updated h_t
        return self.W(h)           # z_{t+1} = W h_t


block_lm = BlockLM(V, d=128, n_h=4)
print("parameters:", num_params(block_lm))
train(block_lm, block_size=64, max_iters=3000, lr=1e-3)
results["one block, no positions"] = estimate_loss(block_lm, block_size=64)["val"]

## 6. Assembling GPT (Chapter 6)

### 6.1 Why position has to be added by hand

Without the causal mask, attention is permutation-equivariant: permuting the rows of $H$ permutes the rows of the output in the same way, with the same values.
Attention has no notion of order; the causal mask only constrains which pairs may interact.

In [ ]:
torch.manual_seed(0)
T, d_k = 6, 8
H = torch.randn(T, d_k)
perm = torch.randperm(T)
out = attention(H, H, H, causal=False)
out_perm = attention(H[perm], H[perm], H[perm], causal=False)
print("Attention(PH) == P Attention(H):", torch.allclose(out_perm, out[perm], atol=1e-6))

**Exercise 22.** Repeat the comparison above with `causal=True`. Is causal attention permutation-equivariant?
Does the causal mask alone give the model a useful notion of position, such as "the previous token"? Explain.

In [ ]:
# TODO (Exercise 22): compare attention(PH, PH, PH) with P attention(H, H, H) for causal=True.

> **Answer:** *write your answer here.*

Order is injected with a second, learned embedding table indexed by position, $E_{\mathrm{pos}}\in\mathbb R^{L_{\max}\times d}$, where $L_{\max}$ is the context length (`block_size`).
The input to the first block is
$$h_t^{(0)} = e_{x_t} + p_t,$$
with $p_t$ row $t$ of $E_{\mathrm{pos}}$.

### 6.2 The full model

The forward pass of GPT:
1. embed tokens and positions and add them, $h_t^{(0)} = e_{x_t} + p_t$;
2. pass the result through $L$ blocks, giving $h_t^{(1)},\dots,h_t^{(L)}$;
3. apply a final LayerNorm, $h_t := \mathrm{LayerNorm}(h_t^{(L)})$;
4. project to logits, $z_{t+1} = W_{\mathrm{head}}h_t$.

With **weight tying**, $W_{\mathrm{head}} = E_{\mathrm{tok}}$: the same tensor maps token ids to vectors and vectors back to token scores.
As in `gpt_intro/gpt.py`, all weights are initialized from $\mathcal N(0, 0.02^2)$ and biases at zero.

**Exercise 23.** Complete the model: tie the weights and implement the forward pass.

In [ ]:
class GPT(nn.Module):
    def __init__(self, V, block_size, n_layer, n_h, d, dropout=0.0):
        super().__init__()
        self.block_size = block_size
        self.E_tok = nn.Embedding(V, d)
        self.E_pos = nn.Embedding(block_size, d)
        self.dropout = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([Block(d, n_h, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(d)
        self.W_head = nn.Linear(d, V, bias=False)
        # TODO (Exercise 23a): weight tying, W_head and E_tok must share the same tensor.

        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.zeros_(m.bias)

    def forward(self, idx):
        B, T = idx.shape
        assert T <= self.block_size
        # TODO (Exercise 23b): the four steps of the forward pass listed above.
        # Apply self.dropout to h_t^(0). Return the logits, shape (B, T, V).
        raise NotImplementedError("Exercise 23b")

### 6.3 Parameter count

Per block, the attention projections contribute $4d^2$ parameters and the feed-forward network $8d^2$, so
$$N_{\mathrm{total}} \approx 12\,L\,d^2 + Vd + L_{\max}d,$$
up to the biases and LayerNorm parameters.
For the configuration trained below ($L = 4$, $d = 128$, $V = 65$, $L_{\max} = 128$) the notes quote an exact count of $816{,}512$.

**Exercise 24.** Derive the *exact* number of parameters, including biases and LayerNorm parameters, as a function of $V$, $L_{\max}$, $L$ and $d$, and check it against the model.
(Look at which `nn.Linear` layers have a bias, and remember weight tying.)

In [ ]:
def gpt_param_count(V, block_size, n_layer, d):
    # TODO (Exercise 24): exact number of parameters of GPT(V, block_size, n_layer, n_h, d).
    raise NotImplementedError("Exercise 24")


config = dict(block_size=128, n_layer=4, n_h=4, d=128, dropout=0.1)
gpt = GPT(V, **config)
L, dd, Lmax = config["n_layer"], config["d"], config["block_size"]
print(f"exact:    {num_params(gpt):,}")
print(f"estimate: {12 * L * dd**2 + V * dd + Lmax * dd:,}")
check("head and token embedding are the same tensor", gpt.W_head.weight is gpt.E_tok.weight)
check("count matches the notes (816,512)", num_params(gpt) == 816_512)
check("your formula == num_params", gpt_param_count(V, Lmax, L, dd) == num_params(gpt))
check("forward pass shape", gpt(torch.zeros(2, 10, dtype=torch.long)).shape == (2, 10, V))

> **Answer:** *write your answer here.*

## 7. Training and generation (Chapter 7)

### 7.1 Training the full GPT

The same `train` loop as before, with the configuration of `scripts/train_gpt.py`: 2000 steps of AdamW with learning rate $3\times10^{-4}$, batches of 64 chunks of 128 tokens, and dropout 0.1.
On a GPU this takes about a minute; on a CPU, about 15–20 minutes (reduce `max_iters` for a quicker, worse model).

In [ ]:
max_iters = 2000
history = train(gpt, block_size=config["block_size"], max_iters=max_iters, lr=3e-4, eval_interval=200)
results["GPT"] = history[-1][2]

its, tr, va = zip(*history)
plt.figure(figsize=(6, 3.5))
plt.plot(its, tr, "o-", label="train")
plt.plot(its, va, "o-", label="val")
plt.axhline(results["counting bigram"], color="gray", ls="--", label="counting bigram (val)")
plt.xlabel("step")
plt.ylabel("loss (nats/token)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

**Exercise 25.** Compare the training and validation curves. Is the model overfitting?
Compare its validation loss with the best $n$-gram of Exercise 9 (about $1.95$ nats at $n = 4$), which also has a fixed horizon.
Why can the GPT see $128$ tokens of context without suffering from the sparsity that ruins the $n$-gram models for large $n$?

> **Answer:** *write your answer here.*

### 7.2 Generation

Generation samples from $p(x_{t+1}\mid x_{\le t})$, appends the sample, and repeats, cropping the input to the last $L_{\max}$ tokens.
Two adjustments to the distribution are standard:

* **temperature** $\tau$: sample from $\sigma(z_{t+1}/\tau)$; $\tau<1$ sharpens the distribution, $\tau>1$ flattens it;
* **top-$k$**: only the $k$ most likely tokens can be sampled.

In [ ]:
for temperature, top_k in [(1.0, None), (0.8, 20), (1.5, None)]:
    print(f"===== temperature {temperature}, top_k {top_k} =====")
    print(generate(gpt, 400, block_size=config["block_size"], prompt="ROMEO:\n",
                   temperature=temperature, top_k=top_k))
    print()

**Exercise 26.** What distribution is sampled in the limits $\tau \to 0$ and $\tau \to \infty$?
What does `top_k=1` do, and why does it tend to produce repetitive text?

> **Answer:** *write your answer here.*

## Summary

Validation loss of every model, from the uniform baseline to the full GPT.
The losses of the counting bigram and the uniform model are exact; those of the trained models are estimated on random validation batches, and fluctuate by about $\pm0.01$ between runs.
Each step of the notes changes only how the context is summarized before the map to logits; the loss, the training loop and the sampling procedure stayed the same throughout.

In [ ]:
for name, loss in results.items():
    print(f"{name:25s} {loss:.4f} nats/token   perplexity {math.exp(loss):6.2f}")

plt.figure(figsize=(7, 3.5))
plt.barh(list(results), list(results.values()))
plt.gca().invert_yaxis()
plt.xlabel("validation loss (nats/token)")
plt.grid(alpha=0.3, axis="x")
plt.show()